### 1. roboflow 서버의 모델을 통한 예측

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)  # .env 파일 읽기

rf_api_key = os.getenv("RF_API_KEY")

# print(rf_api_key)

In [ ]:
from inference_sdk import InferenceHTTPClient
CLIENT = InferenceHTTPClient(
    api_url="https://detect.roboflow.com",
    api_key= rf_api_key
)
filename = './images/number1.jpg'
result = CLIENT.infer(filename, model_id="-i4jas/numbers-td7zg-2-yolov8n-t1")
print(result, "\n")

from pprint import pprint
# pprint(result['predictions'])
pprint([result['predictions'][i]['class'] for i in range(len(result['predictions']))])

### 바운딩 박스 그리기

In [ ]:
import cv2
filename = './images/number1.jpg'
img = cv2.imread(filename)
for pred in result["predictions"]:
    x, y = pred['x'], pred['y']
    width, height = pred['width'], pred['height']
    conf = pred['confidence']
    cls = pred['class']
    x1, y1 = int(x-width/2), int(y-height/2)
    x2, y2 = int(x+width/2), int(y+height/2)
    cv2.rectangle(img, (x1,y1), (x2, y2), (0,0,255), 2)
    cv2.putText(img, f'{cls} {conf:.4f}', (x1,y1), cv2.FONT_HERSHEY_PLAIN, 1, (0,0,255))

cv2.imshow('image', img)
cv2.waitKey(0)    
cv2.destroyAllWindows()

###  roboflow 패키지의 Model 객체를 통한 예측
### ./images/number2.jpg 추론하기

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=rf_api_key)

# 기본 워크스페이스 조회 → 그 안에서 "numbers-doqnw" 프로젝트 선택
project = rf.workspace().project("numbers-td7zg")

# 버전 3의 학습된 모델 객체 (confidence=40, overlap=30, format='json' 기본값)
model = project.version(3).model

# 추론할 로컬 이미지 경로
filename = './images/number2.jpg'

# 이미지를 base64로 인코딩해 Roboflow 서버로 POST → 결과를 dict로 변환
result = model.predict(filename).json()

### 바운딩 박스 그리기

In [ ]:

import cv2

# 이미지 파일 읽기
img = cv2.imread(filename)

# 이미지의 세로(height), 가로(width) 추출
img_height, img_width = img.shape[:2]

# YOLO 모델이 640x640 기준으로 예측했을 경우 스케일 비율 계산
x_ratio, y_ratio = 640 / img_width, 640 / img_height
print(img_width, img_height, x_ratio, y_ratio)

# 예측된 객체(predictions) 반복 처리
for pred in result["predictions"]:

    # 중심 좌표(x, y), 박스 크기(width, height)
    x, y = pred['x'], pred['y']
    w, h = pred['width'], pred['height']

    # 클래스 이름(class), 신뢰도(confidence)
    cls, conf = pred['class'], pred['confidence']

    # YOLO 형식(center x, center y, width, height)을
    # 좌측상단(x1,y1), 우측하단(x2,y2) 좌표로 변환
    x1 = int(x - w/2)
    y1 = int(y - h/2)
    x2 = int(x + w/2)
    y2 = int(y + h/2)

    # 바운딩박스 그리기
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 2)

    # 클래스명 + 신뢰도 표시
    cv2.putText(
        img, 
        f'{cls} {conf:.4f}',    # ex) digit 0.9876
        (x1, y1),               # 텍스트 위치 = 박스 좌상단
        cv2.FONT_HERSHEY_PLAIN, # 폰트
        1,                      # 폰트 크기
        (0, 0, 255)             # 빨간색
    )

# 이미지 출력 (창 닫기 전까지 대기)
cv2.imshow('image', img)
cv2.waitKey(0)

# 모든 OpenCV 창 닫기
cv2.destroyAllWindows()

### ./images/number1.jpg 추론하기

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=rf_api_key)
project = rf.workspace().project("numbers-doqnw")
model = project.version(3).model

filename = './images/number1.jpg'
result = model.predict(filename).json()

### 바운딩박스 그리기

In [ ]:
import cv2

img = cv2.imread(filename)
img_height, img_width = img.shape[:2]
x_ratio, y_ratio = 640/img_width, 640/img_height
print(img_width, img_height, x_ratio, y_ratio)
for pred in result["predictions"]:
    # 중심 좌표(x, y), 박스 크기(width, height)
    x, y = pred['x'], pred['y']
    width, height = pred['width'], pred['height']

    # 클래스 이름(class), 신뢰도(confidence)
    conf, cls = pred['confidence'], pred['class']

    # YOLO 형식(center x, center y, width, height)을
    # 좌측상단(x1,y1), 우측하단(x2,y2) 좌표로 변환
    x1, y1 = int(x-width/2), int(y-height/2)
    x2, y2 = int(x+width/2), int(y+height/2)
    x1, x2 = int(x1*x_ratio), int(x2*x_ratio)
    y1, y2 = int(y1*y_ratio), int(y2*y_ratio)
    
     # 바운딩박스 그리기
    cv2.rectangle(img, (x1,y1), (x2, y2), (0,0,255), 2)

    # 클래스명 + 신뢰도 표시
    cv2.putText(img, 
                f'{cls} {conf:.4f}',     # ex) digit 0.9876
                (x1,y1),                 # 텍스트 위치 = 박스 좌상단
                cv2.FONT_HERSHEY_PLAIN,  # 폰트
                1,                       # 폰트 크기
                (0,0,255))               # 빨간색 (BGR)
cv2.imshow('image', img)
cv2.waitKey(0)
cv2.destroyAllWindows()